# Clean, scale and encode: the rules in code

**Optional code walkthrough for AI 101 (Module 5).** Not graded. Quiz scores do not depend on it.

**Objective.** Reproduce the worked examples from Lessons 5.1 and 5.2 in Python: mean imputation, min-max scaling, standardization and one-hot encoding. Then see why the stored training statistics, not the test data, must be used to transform test values.

**Supports.** Lesson 5.1 (section “Mean-imputation example”), Lesson 5.2 (sections “State the scaling rule”, “Standardization uses training statistics” and “Encode categories without false magnitude”) and Lesson 5.3 (section “Split, fit and apply consistently”).

Run the cells from top to bottom. All data are synthetic and invented for teaching.


## About the data

All examples reuse the numbers from the lessons: the readings [20, 22, missing, 24], the bounds 0 to 100 °C with x = 25 °C, and a training mean of 40 °C with a training standard deviation of 10 °C.

The data are illustrative and synthetic. The lessons do not give a unit for the first list, so treat it as arithmetic only.

```
Readings:            20, 22, missing, 24
Scaling bounds:      0 to 100 °C, x = 25 °C
Training statistics: mean 40 °C, standard deviation 10 °C
Categories:          A, B, C
```


## Step 1. Mean imputation

Lesson 5.1 fills the missing reading with the mean of the observed values: (20 + 22 + 24) / 3 = 22. The code keeps a record of which position was changed, because Lesson 5.1 says every change must be documented.

In [ ]:
# Step 1: mean imputation of one missing reading
readings = [20, 22, None, 24]     # None = missing

observed = []
for value in readings:
    if value is not None:
        observed.append(value)
mean_value = sum(observed) / len(observed)
print("Mean of the observed values:", mean_value)

filled = []
changed_positions = []
for position, value in enumerate(readings):
    if value is None:
        filled.append(mean_value)
        changed_positions.append(position)
    else:
        filled.append(value)
print("After imputation:", filled)
print("Changed positions:", changed_positions)

**Expected output**

```
Mean of the observed values: 22.0
After imputation: [20, 22, 22.0, 24]
Changed positions: [2]
```

The missing value became 22, as in Lesson 5.1.  The filled value is an estimate. It does not recover the original measurement. For a predictive pipeline, the mean would come from training data only.

## Step 2. Min-max scaling

The rule in Lesson 5.2 is x_scaled = (x − x_min) / (x_max − x_min). All three values are in °C, so the result has no unit.

In [ ]:
# Step 2: min-max scaling with stated bounds
x = 25.0                  # observation in °C
x_min = 0.0               # stated lower bound in °C
x_max = 100.0             # stated upper bound in °C
x_scaled = (x - x_min) / (x_max - x_min)
print("Min-max scaled value:", x_scaled)

**Expected output**

```
Min-max scaled value: 0.25
```

25 °C becomes 0.25, as in Lesson 5.2. The bounds 0 and 100 °C map to 0 and 1. A future value outside those bounds would map outside 0 to 1.

## Step 3. One-hot encoding

Lesson 5.2 says to avoid coding categories A, B and C as 1, 2 and 3, because that adds an order and distances that do not exist. One-hot encoding uses one indicator column per known category.

In [ ]:
# Step 3: one-hot encoding for three known categories
categories = ["A", "B", "C"]

def one_hot(label):
    row = []
    for category in categories:
        row.append(1 if label == category else 0)
    return row

for label in ["A", "B", "C"]:
    print(label, "->", one_hot(label))

**Expected output**

```
A -> [1, 0, 0]
B -> [0, 1, 0]
C -> [0, 0, 1]
```

These are the same rows as the lesson’s example. The indicators show membership. They carry no physical size. A category that was not in the list would give [0, 0, 0] here. Lesson 5.2 asks you to decide this rule on purpose.

## Step 4. Standardize with stored training statistics

Lesson 5.2 gives z = (x − μ_train) / σ_train. Lesson 5.3 says to apply the stored training statistics to validation and test inputs and not to recalculate them from those sets. The code first does it correctly. Then it shows what happens if you wrongly use the test mean.

In [ ]:
# Step 4: standardization with stored training statistics
train_mean = 40.0         # mean of the training data in °C (stored)
train_std = 10.0          # standard deviation of the training data in °C (stored)
test_values = [60.0, 25.0]    # new inputs in °C

print("Correct: use the stored training statistics")
for x in test_values:
    z = (x - train_mean) / train_std
    print(f"  x = {x:.0f} °C -> z = {z:.2f}")

wrong_mean = sum(test_values) / len(test_values)   # WRONG: statistic taken from the test set
print("Wrong: mean recomputed from the test values =", wrong_mean)
for x in test_values:
    z = (x - wrong_mean) / train_std
    print(f"  x = {x:.0f} °C -> z = {z:.2f}")

**Expected output**

```
Correct: use the stored training statistics
  x = 60 °C -> z = 2.00
  x = 25 °C -> z = -1.50
Wrong: mean recomputed from the test values = 42.5
  x = 60 °C -> z = 1.75
  x = 25 °C -> z = -1.75
```

The correct values match Lesson 5.2: z = 2 for 60 °C and z = −1.5 for 25 °C. With the wrong mean, the same 60 °C gives 1.75. The numbers changed because the test set leaked into the preprocessing. Lesson 5.3 says this crosses the evaluation boundary even if no test target is used. The z values have no unit. Keep the original °C values for physical meaning.

## Try it

Change a value and observe. In the Step 4 code, change `train_mean = 40.0` to `train_mean = 45.0`. Then run Step 4 again.

**Expected result**

```
Correct: use the stored training statistics
  x = 60 °C -> z = 1.50
  x = 25 °C -> z = -2.00
Wrong: mean recomputed from the test values = 42.5
  x = 60 °C -> z = 1.75
  x = 25 °C -> z = -1.75
```

The correct z values change to 1.50 and −2.00, because the stored training statistics changed. The wrong values stay the same, because they never use the stored mean. Stored statistics decide how every later input is transformed. If you change them, you must record that change. If the training data changes, you must also refit the transformation on the training data only.

## Limits

- The values are the lessons’ teaching numbers. They are not sensor data.
- The code does not decide how to treat an unseen category, a constant feature or a very different unit. Lesson 5.2 says you must state those rules.
- Mean imputation is only one rule. Lesson 5.1 lists others and says to choose from the measurement context.
- In a real project, learned preprocessing is fitted inside the training part of each split. This walkthrough only shows the arithmetic.

Lab 1 audits flagged rows and units. Lab 2 uses a split and a physical feature.